# Collect image variants → ZIP → download

For each input image name (e.g. `13898869.png` or just `13898869`), this notebook finds the
`_orignal`, `_minibyte_final` and `_after` variants under a search path, zips them and
downloads the zip.

**Steps:** run cell 1 (config), cell 2 (input list), cell 3 (collect), cell 4 (zip + download).

## 0. (Optional) Mount Google Drive
Skip if your images are already on the Colab filesystem.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Config

In [ ]:
#@title Config { display-mode: "form" }

SEARCH_PATH = "/content/drive/MyDrive/autofox/results"  #@param {type:"string"}
RECURSIVE = True  #@param {type:"boolean"}
ZIP_NAME = "variants.zip"  #@param {type:"string"}
# Keep the folder structure of each match inside the zip (relative to SEARCH_PATH)
PRESERVE_TREE = False  #@param {type:"boolean"}

# Variants to pull for every input image. Edit freely.
SUFFIXES = ["_orignal", "_minibyte_final", "_after"]

# Extensions to consider when the suffixed file's extension is unknown.
EXTENSIONS = [".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"]

import os
assert os.path.isdir(SEARCH_PATH), f"Not a directory: {SEARCH_PATH}"
print("OK ->", SEARCH_PATH)

## 2. Input images

Pick **one** of the three options below (run only the one you want).
Names may be bare stems (`13898869`), filenames (`13898869.png`) or full paths —
only the stem is used, and a known variant suffix on it is stripped automatically.

In [ ]:
#@title 2a. Paste a list (one per line, or comma separated)
raw = """
17558544
17501917
17435451
17435292
17435144
17317432
17222597
17191223
17187496
17184574
17147732
17124658
"""

import re
INPUTS = [t.strip() for t in re.split(r"[\n,]", raw) if t.strip()]
print(len(INPUTS), "inputs:", INPUTS)

In [ ]:
#@title 2b. Upload the input images from your machine (names only are used)
from google.colab import files
uploaded = files.upload()
INPUTS = list(uploaded.keys())
print(len(INPUTS), "inputs:", INPUTS)

In [ ]:
#@title 2c. Take every image in a folder as the input list
INPUT_DIR = "/content/drive/MyDrive/autofox/inputs"  #@param {type:"string"}

import os
INPUTS = sorted(
    f for f in os.listdir(INPUT_DIR)
    if os.path.splitext(f)[1].lower() in EXTENSIONS
)
print(len(INPUTS), "inputs:", INPUTS[:10], "..." if len(INPUTS) > 10 else "")

## 3. Collect the variants

In [ ]:
import os
from collections import defaultdict


def build_index(root, recursive=True):
    """Map lowercase filename stem -> list of full paths."""
    index = defaultdict(list)
    if recursive:
        walker = os.walk(root)
    else:
        walker = [(root, [], os.listdir(root))]
    for dirpath, _dirnames, filenames in walker:
        for fn in filenames:
            stem, ext = os.path.splitext(fn)
            if ext.lower() in EXTENSIONS:
                index[stem.lower()].append(os.path.join(dirpath, fn))
    return index


def base_stem(name):
    """'13898869_after.png' -> '13898869'"""
    stem = os.path.splitext(os.path.basename(name))[0]
    for suf in sorted(SUFFIXES, key=len, reverse=True):
        if stem.lower().endswith(suf.lower()):
            return stem[: -len(suf)]
    return stem


index = build_index(SEARCH_PATH, RECURSIVE)
print(f"Indexed {sum(len(v) for v in index.values())} images under {SEARCH_PATH}\n")

found, missing, ambiguous = [], [], []

for item in INPUTS:
    stem = base_stem(item)
    for suf in SUFFIXES:
        key = (stem + suf).lower()
        matches = index.get(key, [])
        if not matches:
            missing.append(stem + suf)
        else:
            if len(matches) > 1:
                ambiguous.append((stem + suf, matches))
            found.append(matches[0])

# de-duplicate, keep order
found = list(dict.fromkeys(found))

print(f"Found   : {len(found)} files")
print(f"Missing : {len(missing)}")
for m in missing:
    print("   -", m)
if ambiguous:
    print(f"\nAmbiguous (first match used): {len(ambiguous)}")
    for name, paths in ambiguous:
        print("   -", name, "->", paths)

## 4. Zip and download

In [ ]:
import os, zipfile
from google.colab import files

assert found, "Nothing to zip — check SEARCH_PATH / SUFFIXES / INPUTS."

zip_path = os.path.join("/content", ZIP_NAME)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in found:
        if PRESERVE_TREE:
            arcname = os.path.relpath(path, SEARCH_PATH)
        else:
            arcname = os.path.basename(path)
        zf.write(path, arcname)

size_mb = os.path.getsize(zip_path) / 1e6
print(f"{zip_path}  ({len(found)} files, {size_mb:.1f} MB)")

files.download(zip_path)

In [ ]:
#@title (Optional) Save the zip to Drive instead of downloading
DRIVE_DEST = "/content/drive/MyDrive"  #@param {type:"string"}

import shutil, os
shutil.copy(zip_path, os.path.join(DRIVE_DEST, os.path.basename(zip_path)))
print("Copied to", DRIVE_DEST)